# Data Cleaning and Integration

This notebook prepares the datasets explored in the previous notebooks for the analysis of electric vehicle adoption across Europe.

The process includes standardizing country names and time periods, selecting relevant variables, handling missing values, and integrating EV adoption data with charging infrastructure and economic indicators.

The objective is to create a clean, consistent, and analysis-ready dataset while preserving the original raw data.

In [4]:
import pandas as pd
import numpy as np

## Load Raw Datasets

The original datasets are loaded separately to preserve their structure and allow cleaning operations to be performed on working copies.

In [ ]:
# Load raw datasets
ember = pd.read_csv("../data/raw/ember_monthly_ev_sales.csv")

registrations = pd.read_csv("../data/raw/eurostat_vehicle_registrations.csv")

charging = pd.read_csv("../data/raw/eurostat_charging_infrastructure.csv")

gdp = pd.read_csv("../data/raw/eurostat_gdp_per_capita.csv")

population = pd.read_csv("../data/raw/eurostat_population.csv")

electricity = pd.read_csv("../data/raw/eurostat_electricity_prices.csv")

In [ ]:
# Check dataset dimensions
datasets = {
    "Ember": ember,
    "Registrations": registrations,
    "Charging": charging,
    "GDP": gdp,
    "Population": population,
    "Electricity": electricity,
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Ember: (46681, 24)
Registrations: (397, 11)
Charging: (488, 10)
GDP: (883, 10)
Population: (581, 9)
Electricity: (1491, 13)


## Ember – EV Adoption Data

This section prepares the Ember dataset, which provides monthly battery electric vehicle (BEV) registrations and registration shares across European countries.

### Selecting the EV Adoption Indicator

Battery Electric Vehicles (BEVs) are selected as the primary adoption indicator because they represent fully electric vehicles.

Plug-in Hybrid Electric Vehicles (PHEVs) are excluded from the main analysis but may be considered in a supplementary comparison.

In [ ]:
# Inspect available drivetrain categories
ember["Drivetrain"].value_counts()

Drivetrain
Battery electric               6362
Electric vehicle (BEV+PHEV)    6362
All drivetrains                6362
Plug-in hybrid                 5685
Hybrid                         5662
Petrol                         4992
Diesel                         4992
Other                          4638
Combustion (unspecified)       1370
Ethanol/petrol blend            256
Name: count, dtype: int64

In [9]:
# Create a working copy of the raw Ember dataset
ember_clean = ember.copy()

# Keep only Battery Electric Vehicle registrations
ember_clean = ember_clean[ember_clean["Drivetrain"] == "Battery electric"].copy()

# Check the result
print("Shape:", ember_clean.shape)
print("Drivetrains:", ember_clean["Drivetrain"].unique())

Shape: (6362, 24)
Drivetrains: <ArrowStringArray>
['Battery electric']
Length: 1, dtype: str


In [10]:
# Select relevant columns
ember_clean = ember_clean[
    [
        "Area",
        "ISO 3 code",
        "Date",
        "Registrations",
        "Share of registrations (%)",
        "Continent",
        "Ember region",
        "EU member",
    ]
].copy()

# Rename columns for consistency
ember_clean = ember_clean.rename(
    columns={
        "Area": "country",
        "ISO 3 code": "country_code",
        "Date": "date",
        "Registrations": "bev_registrations",
        "Share of registrations (%)": "bev_share_pct",
        "Continent": "continent",
        "Ember region": "ember_region",
        "EU member": "eu_member",
    }
)

# Convert date to datetime
ember_clean["date"] = pd.to_datetime(ember_clean["date"])

# Extract year
ember_clean["year"] = ember_clean["date"].dt.year

ember_clean.head()

,country,country_code,date,bev_registrations,bev_share_pct,continent,ember_region,eu_member,year
0,Albania,ALB,2019-01-01,1.0,0.03,Europe,Europe,False,2019
8,Albania,ALB,2019-02-01,1.0,0.03,Europe,Europe,False,2019
16,Albania,ALB,2019-03-01,3.0,0.17,Europe,Europe,False,2019
24,Albania,ALB,2019-04-01,2.0,0.08,Europe,Europe,False,2019
32,Albania,ALB,2019-05-01,0.0,0.00,Europe,Europe,False,2019


In [11]:
ember_clean["continent"].value_counts(dropna=False)

continent
Europe           3630
Asia             1576
South America     504
North America     356
NaN               174
Oceania           108
Africa             14
Name: count, dtype: int64

In [ ]:
# Keep European observations
ember_clean = ember_clean[ember_clean["continent"] == "Europe"].copy()

print("Shape:", ember_clean.shape)
print("Countries:", ember_clean["country"].nunique())

Shape: (3630, 9)
Countries: 33


In [13]:
# Check missing values
ember_clean.isna().sum()

country              0
country_code         0
date                 0
bev_registrations    0
bev_share_pct        0
continent            0
ember_region         0
eu_member            0
year                 0
dtype: int64

In [14]:
# Check duplicate country-date observations
ember_clean.duplicated(subset=["country", "date"]).sum()

np.int64(0)

In [15]:
# Check date coverage
print("Start:", ember_clean["date"].min())
print("End:", ember_clean["date"].max())

Start: 2005-01-01 00:00:00
End: 2026-10-01 00:00:00


In [16]:
# Save cleaned Ember data
ember_clean.to_csv("../data/cleaned/ember_bev_clean.csv", index=False)

## Charging Infrastructure – Eurostat

This section prepares the Eurostat charging infrastructure dataset for integration with EV adoption data.

### Selecting the Charging Infrastructure Indicator

The main charging infrastructure indicator is the number of recharging points, measured as a count.

Other infrastructure categories and indicators measured in kilowatts are excluded from the primary analysis because they represent different concepts.

Heavy-duty charging infrastructure is excluded because the project focuses on passenger electric vehicles.

In [17]:
# Create a working copy
charging_clean = charging.copy()

# Keep passenger-car recharging points measured as counts
charging_clean = charging_clean[
    (charging_clean["tra_infr"] == "Recharging point")
    & (charging_clean["unit"] == "Number")
].copy()

# Select relevant columns
charging_clean = charging_clean[["geo", "TIME_PERIOD", "OBS_VALUE", "OBS_FLAG"]].copy()

# Rename columns
charging_clean = charging_clean.rename(
    columns={
        "geo": "country",
        "TIME_PERIOD": "year",
        "OBS_VALUE": "charging_points",
        "OBS_FLAG": "charging_flag",
    }
)

# Convert year to integer
charging_clean["year"] = charging_clean["year"].astype(int)

# Check result
print("Shape:", charging_clean.shape)
print("Countries/regions:", charging_clean["country"].nunique())
charging_clean.head()

Shape: (158, 4)
Countries/regions: 33


,country,year,charging_points,charging_flag
0,Austria,2021,9143.0,NaN
1,Austria,2022,14526.0,NaN
2,Austria,2023,19663.0,NaN
3,Austria,2024,25632.0,NaN
4,Austria,2025,34333.0,NaN


## GDP per Capita – Eurostat

This section prepares GDP per capita as an indicator of economic development and purchasing power.

### Selecting Relevant Columns

Only country, year, GDP per capita, and observation quality information are retained.

In [ ]:
# Create a working copy of the GDP dataset
gdp_clean = gdp.copy()

# Select relevant columns
gdp_clean = gdp_clean[["geo", "TIME_PERIOD", "OBS_VALUE", "OBS_FLAG"]].copy()

# Rename columns
gdp_clean = gdp_clean.rename(
    columns={
        "geo": "country",
        "TIME_PERIOD": "year",
        "OBS_VALUE": "gdp_per_capita",
        "OBS_FLAG": "gdp_flag",
    }
)

# Convert year to integer
gdp_clean["year"] = gdp_clean["year"].astype(int)

gdp_clean.head()

,country,year,gdp_per_capita,gdp_flag
0,Albania,2005,2880,NaN
1,Albania,2006,3080,NaN
2,Albania,2007,3300,NaN
3,Albania,2008,3550,NaN
4,Albania,2009,3680,NaN


In [ ]:
# Check dataset dimensions
print("Shape:", gdp_clean.shape)

# Check country and year coverage
print("Countries/regions:", gdp_clean["country"].nunique())
print("Year range:", gdp_clean["year"].min(), "-", gdp_clean["year"].max())

# Check duplicate country-year observations
print(
    "Duplicate country-year rows:",
    gdp_clean.duplicated(subset=["country", "year"]).sum(),
)

# Check missing values
gdp_clean[["country", "year", "gdp_per_capita"]].isna().sum()

Shape: (883, 4)
Countries/regions: 43
Year range: 2005 - 2025
Duplicate country-year rows: 0


country           0
year              0
gdp_per_capita    0
dtype: int64

In [22]:
# Inspect country names
sorted(gdp_clean["country"].unique())

['Albania',
 'Austria',
 'Belgium',
 'Bosnia and Herzegovina',
 'Bulgaria',
 'Croatia',
 'Cyprus',
 'Czechia',
 'Denmark',
 'Estonia',
 'Euro area (EA11-1999, EA12-2001, EA13-2007, EA15-2008, EA16-2009, EA17-2011, EA18-2014, EA19-2015, EA20-2023, EA21-2026)',
 'Euro area - 12 countries (2001-2006)',
 'Euro area - 19 countries  (2015-2022)',
 'Euro area – 20 countries (2023-2025)',
 'Euro area – 21 countries (from 2026)',
 'European Union - 27 countries (from 2020)',
 'Finland',
 'France',
 'Germany',
 'Greece',
 'Hungary',
 'Iceland',
 'Ireland',
 'Italy',
 'Latvia',
 'Lithuania',
 'Luxembourg',
 'Malta',
 'Moldova',
 'Montenegro',
 'Netherlands',
 'North Macedonia',
 'Norway',
 'Poland',
 'Portugal',
 'Romania',
 'Serbia',
 'Slovakia',
 'Slovenia',
 'Spain',
 'Sweden',
 'Switzerland',
 'Türkiye']

### GDP Dataset – Cleaning Summary

The GDP dataset is reduced to country, year, GDP per capita, and observation flags.

The dataset contains annual observations and is structured at country-year level.

Country names and geographic coverage will be standardized during the integration stage to ensure consistency with the EV adoption data.

In [23]:
# Check the cleaned GDP dataset
gdp_clean.head()

,country,year,gdp_per_capita,gdp_flag
0,Albania,2005,2880,NaN
1,Albania,2006,3080,NaN
2,Albania,2007,3300,NaN
3,Albania,2008,3550,NaN
4,Albania,2009,3680,NaN


## Population – Eurostat

This section prepares population data to account for differences in country size.

### Selecting Relevant Columns

Only country, year, population, and observation quality information are retained.

In [25]:
# Create a working copy of the population dataset
population_clean = population.copy()

# Select relevant columns
population_clean = population_clean[
    ["geo", "TIME_PERIOD", "OBS_VALUE", "OBS_FLAG"]
].copy()

# Rename columns
population_clean = population_clean.rename(
    columns={
        "geo": "country",
        "TIME_PERIOD": "year",
        "OBS_VALUE": "population",
        "OBS_FLAG": "population_flag",
    }
)

# Convert year to integer
population_clean["year"] = population_clean["year"].astype(int)

population_clean.head()

,country,year,population,population_flag
0,Andorra,2016,71732,NaN
1,Andorra,2018,74794,e
2,Andorra,2019,76177,NaN
3,Andorra,2020,77543,e
4,Andorra,2022,79535,NaN


In [ ]:
# Check dataset dimensions
print("Shape:", population_clean.shape)

# Check country and year coverage
print("Countries/regions:", population_clean["country"].nunique())
print(
    "Year range:", population_clean["year"].min(), "-", population_clean["year"].max()
)

# Check duplicate country-year observations
print(
    "Duplicate country-year rows:",
    population_clean.duplicated(subset=["country", "year"]).sum(),
)

# Check missing values in main analytical columns
population_clean[["country", "year", "population"]].isna().sum()

Shape: (581, 4)
Countries/regions: 51
Year range: 2015 - 2026
Duplicate country-year rows: 0


country       0
year          0
population    0
dtype: int64

In [27]:
# Inspect country names
sorted(population_clean["country"].unique())

['Albania',
 'Andorra',
 'Armenia',
 'Austria',
 'Azerbaijan',
 'Belarus',
 'Belgium',
 'Bosnia and Herzegovina',
 'Bulgaria',
 'Croatia',
 'Cyprus',
 'Czechia',
 'Denmark',
 'Estonia',
 'Euro area – 20 countries (2023-2025)',
 'Euro area – 21 countries (from 2026)',
 'European Union - 27 countries (from 2020)',
 'Finland',
 'France',
 'Georgia',
 'Germany',
 'Greece',
 'Hungary',
 'Iceland',
 'Ireland',
 'Italy',
 'Kosovo*',
 'Latvia',
 'Liechtenstein',
 'Lithuania',
 'Luxembourg',
 'Malta',
 'Moldova',
 'Monaco',
 'Montenegro',
 'Netherlands',
 'North Macedonia',
 'Norway',
 'Poland',
 'Portugal',
 'Romania',
 'San Marino',
 'Serbia',
 'Slovakia',
 'Slovenia',
 'Spain',
 'Sweden',
 'Switzerland',
 'Türkiye',
 'Ukraine',
 'United Kingdom']

### Population Dataset – Cleaning Summary

The population dataset is reduced to country, year, population, and observation flags.

The dataset contains annual population observations measured on 1 January.

Country names and geographic coverage will be standardized during the integration stage to ensure consistency with the EV adoption data.

In [28]:
# Inspect cleaned population data
population_clean.head()

,country,year,population,population_flag
0,Andorra,2016,71732,NaN
1,Andorra,2018,74794,e
2,Andorra,2019,76177,NaN
3,Andorra,2020,77543,e
4,Andorra,2022,79535,NaN


## Electricity Prices – Eurostat

This section prepares household electricity prices as a proxy for electricity costs across European countries.

### Selecting Relevant Columns

Only country, reporting period, electricity price, and observation quality information are retained.

In [ ]:
# Create a working copy of the electricity dataset
electricity_clean = electricity.copy()

# Select relevant columns
electricity_clean = electricity_clean[
    ["geo", "TIME_PERIOD", "OBS_VALUE", "OBS_FLAG"]
].copy()

# Rename columns
electricity_clean = electricity_clean.rename(
    columns={
        "geo": "country",
        "TIME_PERIOD": "period",
        "OBS_VALUE": "electricity_price",
        "OBS_FLAG": "electricity_flag",
    }
)

electricity_clean.head()

,country,period,electricity_price,electricity_flag
0,Albania,2011-S1,0.1152,NaN
1,Albania,2011-S2,0.1157,NaN
2,Albania,2012-S1,0.1163,NaN
3,Albania,2012-S2,0.1167,NaN
4,Albania,2013-S1,0.1156,NaN


### Aggregating Half-Yearly Prices to Annual Values

Electricity prices are reported twice per year.

The two half-yearly observations are averaged to obtain an annual price for each country. The number of available periods is retained to identify years with incomplete coverage.

The indicator represents household electricity prices, including taxes and levies, rather than public charging prices.

In [ ]:
# Extract year from the half-yearly period
electricity_clean["year"] = electricity_clean["period"].str[:4].astype(int)

# Aggregate half-yearly prices to annual averages
electricity_clean = electricity_clean.groupby(["country", "year"], as_index=False).agg(
    electricity_price=("electricity_price", "mean"),
    electricity_periods=("period", "count"),
)

electricity_clean.head()

,country,year,electricity_price,electricity_periods
0,Albania,2011,0.11545,2
1,Albania,2012,0.11650,2
2,Albania,2013,0.11550,2
3,Albania,2014,0.11580,2
4,Albania,2015,0.08155,2


In [ ]:
# Check dataset dimensions
print("Shape:", electricity_clean.shape)

# Check country and year coverage
print("Countries/regions:", electricity_clean["country"].nunique())
print(
    "Year range:", electricity_clean["year"].min(), "-", electricity_clean["year"].max()
)

# Check duplicate country-year observations
print(
    "Duplicate country-year rows:",
    electricity_clean.duplicated(subset=["country", "year"]).sum(),
)

# Check missing values
electricity_clean[["country", "year", "electricity_price"]].isna().sum()

Shape: (780, 4)
Countries/regions: 43
Year range: 2007 - 2026
Duplicate country-year rows: 0


country              0
year                 0
electricity_price    0
dtype: int64

In [32]:
# Check how many half-yearly observations are available per country-year
electricity_clean["electricity_periods"].value_counts()

electricity_periods
2    711
1     69
Name: count, dtype: int64

In [33]:
# Inspect the cleaned dataset
electricity_clean.head()

,country,year,electricity_price,electricity_periods
0,Albania,2011,0.11545,2
1,Albania,2012,0.11650,2
2,Albania,2013,0.11550,2
3,Albania,2014,0.11580,2
4,Albania,2015,0.08155,2


## Standardizing Country Names

Country names are standardized across datasets to ensure that observations referring to the same country can be correctly merged.

The Ember dataset is used as the reference for geographic coverage.

In [ ]:
# Identify countries included in the main Ember dataset
ember_countries = set(ember_clean["country"].unique())

# Compare country names across datasets
datasets_to_compare = {
    "Charging": charging_clean,
    "GDP": gdp_clean,
    "Population": population_clean,
    "Electricity": electricity_clean,
}

for name, df in datasets_to_compare.items():
    eurostat_countries = set(df["country"].unique())

    print(f"\n{name}")
    print("Countries in Eurostat but not Ember:")
    print(sorted(eurostat_countries - ember_countries))

    print("Countries in Ember but not Eurostat:")
    print(sorted(ember_countries - eurostat_countries))


Charging
Countries in Eurostat but not Ember:
['Georgia', 'North Macedonia', 'Türkiye']
Countries in Ember but not Eurostat:
['Albania', 'Ukraine', 'United Kingdom']

GDP
Countries in Eurostat but not Ember:
['Bosnia and Herzegovina', 'Euro area (EA11-1999, EA12-2001, EA13-2007, EA15-2008, EA16-2009, EA17-2011, EA18-2014, EA19-2015, EA20-2023, EA21-2026)', 'Euro area - 12 countries (2001-2006)', 'Euro area - 19 countries  (2015-2022)', 'Euro area – 20 countries (2023-2025)', 'Euro area – 21 countries (from 2026)', 'European Union - 27 countries (from 2020)', 'Moldova', 'Montenegro', 'North Macedonia', 'Serbia', 'Türkiye']
Countries in Ember but not Eurostat:
['Ukraine', 'United Kingdom']

Population
Countries in Eurostat but not Ember:
['Andorra', 'Armenia', 'Azerbaijan', 'Belarus', 'Bosnia and Herzegovina', 'Euro area – 20 countries (2023-2025)', 'Euro area – 21 countries (from 2026)', 'European Union - 27 countries (from 2020)', 'Georgia', 'Kosovo*', 'Liechtenstein', 'Moldova', 'Mon

In [ ]:
# Standardize Türkiye to Turkey in Eurostat datasets

country_mapping = {"Türkiye": "Turkey"}

for df in [charging_clean, gdp_clean, population_clean, electricity_clean]:
    df["country"] = df["country"].replace(country_mapping)

### Filtering Geographic Coverage

Ember defines the geographic scope of the analysis.

- Eurostat datasets are restricted to countries included in Ember.
- European countries with unavailable Eurostat indicators are retained in the main dataset and will have missing values for those indicators.
- EU and Euro area aggregates are excluded from country-level analysis.

In [ ]:
# Reference country list from Ember
ember_countries = set(ember_clean["country"].unique())

# Filter Eurostat datasets to Ember countries
charging_clean = charging_clean[charging_clean["country"].isin(ember_countries)].copy()

gdp_clean = gdp_clean[gdp_clean["country"].isin(ember_countries)].copy()

population_clean = population_clean[
    population_clean["country"].isin(ember_countries)
].copy()

electricity_clean = electricity_clean[
    electricity_clean["country"].isin(ember_countries)
].copy()

In [ ]:
# Check remaining country coverage

for name, df in {
    "Charging": charging_clean,
    "GDP": gdp_clean,
    "Population": population_clean,
    "Electricity": electricity_clean,
}.items():

    print(f"\n{name}")
    print("Countries:", df["country"].nunique())
    print(
        "Countries not in Ember:", sorted(set(df["country"].unique()) - ember_countries)
    )


Charging
Countries: 30
Countries not in Ember: []

GDP
Countries: 31
Countries not in Ember: []

Population
Countries: 33
Countries not in Ember: []

Electricity
Countries: 32
Countries not in Ember: []


## Aggregating Ember Data to Annual Values

The Ember dataset contains monthly data, while the Eurostat datasets contain yearly data.

To make them compatible, we combine the monthly data into yearly values:
- Sum monthly BEV registrations to get annual registrations.
- Calculate the average monthly BEV registration share.
- Count how many months of data are available for each country and year.

In [ ]:
ember_annual = ember_clean.groupby(
    ["country", "country_code", "year"], as_index=False
).agg(
    bev_registrations=("bev_registrations", "sum"),
    avg_monthly_bev_share_pct=("bev_share_pct", "mean"),
    months_available=("date", "count"),
)

ember_annual.head()

,country,country_code,year,bev_registrations,avg_monthly_bev_share_pct,months_available
0,Albania,ALB,2019,24.0,0.063333,12
1,Albania,ALB,2020,239.0,0.573333,12
2,Albania,ALB,2021,262.0,0.477500,12
3,Albania,ALB,2022,624.0,1.251667,12
4,Albania,ALB,2023,1671.0,2.505833,12


In [ ]:
print("Shape:", ember_annual.shape)
print("Countries:", ember_annual["country"].nunique())
print("Year range:", ember_annual["year"].min(), "-", ember_annual["year"].max())

print(
    "Duplicate country-year rows:",
    ember_annual.duplicated(subset=["country", "year"]).sum(),
)

ember_annual["months_available"].value_counts().sort_index()

Shape: (311, 6)
Countries: 33
Year range: 2005 - 2026
Duplicate country-year rows: 0


months_available
7       1
8       2
9      27
10      4
12    277
Name: count, dtype: int64

## Preparing Datasets for Integration

Before combining the datasets, we keep only the columns needed for our analysis.

We also remove observation flags, which contain additional information about the data but are not needed for our analysis.

In [ ]:
charging_clean = charging_clean[["country", "year", "charging_points"]].copy()

gdp_clean = gdp_clean[["country", "year", "gdp_per_capita"]].copy()

population_clean = population_clean[["country", "year", "population"]].copy()

electricity_clean = electricity_clean[
    ["country", "year", "electricity_price", "electricity_periods"]
].copy()

In [ ]:
datasets_to_check = {
    "Ember": ember_annual,
    "Charging": charging_clean,
    "GDP": gdp_clean,
    "Population": population_clean,
    "Electricity": electricity_clean,
}

for name, df in datasets_to_check.items():
    duplicates = df.duplicated(subset=["country", "year"]).sum()
    print(f"{name}: {duplicates} duplicate country-year rows")

Ember: 0 duplicate country-year rows
Charging: 0 duplicate country-year rows
GDP: 0 duplicate country-year rows
Population: 0 duplicate country-year rows
Electricity: 0 duplicate country-year rows


## Integrating the Datasets

We combine the cleaned datasets using country and year.

Ember is the main dataset. We add charging infrastructure, GDP, population, and electricity prices to it.

Some values may be missing because the Eurostat datasets do not cover every country and year.

In [ ]:
ev_data = ember_annual.merge(
    charging_clean, on=["country", "year"], how="left", validate="one_to_one"
)

ev_data = ev_data.merge(
    gdp_clean, on=["country", "year"], how="left", validate="one_to_one"
)

ev_data = ev_data.merge(
    population_clean, on=["country", "year"], how="left", validate="one_to_one"
)

ev_data = ev_data.merge(
    electricity_clean, on=["country", "year"], how="left", validate="one_to_one"
)

ev_data.head()

,country,country_code,year,bev_registrations,avg_monthly_bev_share_pct,months_available,charging_points,gdp_per_capita,population,electricity_price,electricity_periods
0,Albania,ALB,2019,24.0,0.063333,12,NaN,4850.0,2862427.0,0.09265,2.0
1,Albania,ALB,2020,239.0,0.573333,12,NaN,4710.0,2845955.0,0.09210,2.0
2,Albania,ALB,2021,262.0,0.477500,12,NaN,5190.0,2829741.0,0.09310,2.0
3,Albania,ALB,2022,624.0,1.251667,12,NaN,5500.0,2793592.0,0.09585,2.0
4,Albania,ALB,2023,1671.0,2.505833,12,NaN,NaN,2761785.0,0.10495,2.0


In [ ]:
print("Shape:", ev_data.shape)
print("Countries:", ev_data["country"].nunique())
print(
    "Duplicate country-year rows:", ev_data.duplicated(subset=["country", "year"]).sum()
)

ev_data.info()

Shape: (311, 11)
Countries: 33
Duplicate country-year rows: 0
<class 'pandas.DataFrame'>
RangeIndex: 311 entries, 0 to 310
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   country                    311 non-null    str    
 1   country_code               311 non-null    str    
 2   year                       311 non-null    int32  
 3   bev_registrations          311 non-null    float64
 4   avg_monthly_bev_share_pct  311 non-null    float64
 5   months_available           311 non-null    int64  
 6   charging_points            118 non-null    float64
 7   gdp_per_capita             254 non-null    float64
 8   population                 260 non-null    float64
 9   electricity_price          272 non-null    float64
 10  electricity_periods        272 non-null    float64
dtypes: float64(7), int32(1), int64(1), str(2)
memory usage: 28.9 KB


## Checking Missing Values

We check how many values are missing in each column.

Missing values are expected because some indicators are only available for certain countries and years.

In [ ]:
missing_values = ev_data.isna().sum()

missing_summary = pd.DataFrame(
    {
        "missing_values": missing_values,
        "missing_percentage": (missing_values / len(ev_data) * 100).round(2),
    }
)

missing_summary

,missing_values,missing_percentage
country,0,0.00
country_code,0,0.00
year,0,0.00
bev_registrations,0,0.00
avg_monthly_bev_share_pct,0,0.00
months_available,0,0.00
charging_points,193,62.06
gdp_per_capita,57,18.33
population,51,16.40
electricity_price,39,12.54


## Creating a Population-Based Adoption Indicator

Countries have different population sizes.

To make comparisons fairer, we calculate BEV registrations per 1,000 inhabitants.

In [46]:
ev_data["bev_registrations_per_1000_people"] = (
    ev_data["bev_registrations"] / ev_data["population"] * 1000
)

## Saving the Final Dataset

We save the integrated dataset so it can be used in the analysis notebook.

In [ ]:
ev_data.to_csv("../data/cleaned/electric_vehicle_data.csv", index=False)

: 